# vke-budgets quickstart

The plumbing for exploring the KEV budget of the 3–5 January 2023 atmospheric river: load MERRA-2,
compute the budget, look at the terms and at the residual. The investigation itself starts at the end.

**Running it.** Open a Jupyter session on a compute node (Sherlock OnDemand, 4 CPUs, 32 GB) and pick the
**vke-budgets** kernel. The kernel sets three environment variables, so that no path is written in this
notebook:

| variable | what |
|---|---|
| `VKE_MERRA2_ROOT` | MERRA-2 root, one directory per year (2023-01-03 … 06 must be there) |
| `VKE_CASE_OUT` | where the computed case is written (about 600 MB) |
| `VKE_AR_MASKS` | optional: directory with the AR masks `detect_AR_IKEV_Lap_v2_<date>.nc` |

**Reading the code alongside.** `terms.py` (one function per term, with its equation), then
`budget.window_budget` (the five steps of a window), then `vertical.py` and `horizontal.py` (the operators).

In [ ]:
import os

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from vke_budgets import budget, io, pipeline
from vke_budgets.sources import merra2

MERRA2_ROOT = os.environ["VKE_MERRA2_ROOT"]
CASE_OUT = os.environ["VKE_CASE_OUT"]
AR_MASKS = os.environ.get("VKE_AR_MASKS")

# The domain of the case (and of the reference's website file), in degrees east 0-360.
DOMAIN = {"lon": (190.0, 260.0), "lat": (15.0, 60.0)}
FRAME = "2023-01-04T13:30"   # one 3-hour window, during landfall in California

## 1. The standard input

`merra2.open_day` turns the five MERRA-2 files of a day into the standard input Dataset (see
`standard.py`). It is lazy: nothing is read until a value is needed.

- `time`: centres of the 3-hour averaging windows; `time_inst`: the instantaneous steps, one more.
  Window *i* runs from `time_inst[i]` to `time_inst[i+1]`.
- `*_inst` on pressure levels (`lev`, Pa; NaN below the ground): used for the tendencies.
- `p q u v phi omega` on the 72 model levels (`mlev`, bottom-up): the window-mean state.
- `du/dv/dqdt_<process>`: MERRA-2's own tendencies by physical process.

In [ ]:
day = merra2.open_day("20230104", MERRA2_ROOT)
day

## 2. One window, computed live

`budget.compute` with `windows=[i]` computes a single window (about 20 s). `keep_3d=True` keeps every term
on the pressure levels as well as its column integral; the two extras are MERRA-2's dynamics tendency
(`dyn_crosscheck`) and the flux-form decomposition (`flux_form`).

Every option is in `budget.BudgetOptions`. To test an idea, change one option (or one input field) and
recompute the window: `window_budget` is a pure function of its inputs.

In [ ]:
options = budget.BudgetOptions(keep_3d=True, flux_form=True, dyn_crosscheck=True)
window = budget.compute(day, options, windows=[4])      # 2023-01-04 13:30
print(sorted(v for v in window.data_vars if window[v].ndim == 4))   # the 3D terms
window["IHAKE"]

## 3. The whole case, from files

Three days of column integrals, written once to `VKE_CASE_OUT` (about 4 minutes on 4 CPUs) and then read
lazily. The first run computes; later runs find the files and skip.

In [ ]:
case_options = budget.BudgetOptions(flux_form=True, dyn_crosscheck=True)
pipeline.run("2023-01-03", "2023-01-05", MERRA2_ROOT, CASE_OUT, case_options, n_workers=3)
case = io.open_budget(os.path.join(CASE_OUT, "budget_*.nc"))
case

## Helpers

`region` cuts out the domain (MERRA-2 longitudes run from −180 to 180; the domain crosses 180°).
`map_panels` draws signed fields side by side with **one shared, symmetric color scale**, so that the
sizes of the terms compare honestly. `window_mask` is the reference's AR mask for a window: the mean of the
instantaneous masks at its two ends, ≥ 0.5.

In [ ]:
def region(da):
    da = da.assign_coords(lon=da["lon"] % 360).sortby("lon")
    return da.sel(lon=slice(*DOMAIN["lon"]), lat=slice(*DOMAIN["lat"]))


def map_panels(fields, title, vmax=None, units="kg s$^{-3}$", ncols=3):
    """Signed fields on a shared symmetric diverging scale."""
    fields = {name: region(f) for name, f in fields.items()}
    if vmax is None:   # robust: 98th percentile of the largest field
        vmax = max(float(np.nanpercentile(np.abs(f), 98)) for f in fields.values())
    nrows = -(-len(fields) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 2.9 * nrows), squeeze=False,
                             subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)})
    for ax, (name, f) in zip(axes.flat, fields.items()):
        mesh = ax.pcolormesh(f["lon"], f["lat"], f, cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                             transform=ccrs.PlateCarree(), shading="auto")
        ax.coastlines(linewidth=0.5, color="0.35")
        ax.set_extent([*DOMAIN["lon"], *DOMAIN["lat"]], crs=ccrs.PlateCarree())
        ax.set_title(name, fontsize=10)
    for ax in axes.flat[len(fields):]:
        ax.remove()
    fig.colorbar(mesh, ax=axes, shrink=0.7, label=units)
    fig.suptitle(title)
    return fig


def window_mask(times, var="IKEV"):
    """AR mask for each window in `times` (None without VKE_AR_MASKS)."""
    if not AR_MASKS:
        return None
    days = sorted({str(t)[:10].replace("-", "") for t in times.values})
    inst = xr.concat([xr.open_dataset(os.path.join(AR_MASKS, f"detect_AR_{var}_Lap_v2_{d}.nc"))["binary_tag"]
                      for d in days], "time").astype(float)
    half = np.timedelta64(90, "m")
    start = inst.reindex(time=times.values - half).assign_coords(time=times.values)
    end = inst.reindex(time=times.values + half).assign_coords(time=times.values)
    return 0.5 * (start + end) >= 0.5

## 4. The atmospheric river

IKEV = ⟨qK⟩ and IVT at the start of the window (instantaneous columns, one-hue scales).

In [ ]:
t0 = case["time_inst"].sel(time_inst=np.datetime64(FRAME) - np.timedelta64(90, "m"))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)})
for ax, name, units in zip(axes, ["IKEV", "IVT"], ["kg s$^{-2}$", "kg m$^{-1}$ s$^{-1}$"]):
    f = region(case[name].sel(time_inst=t0))
    mesh = ax.pcolormesh(f["lon"], f["lat"], f, cmap="Blues", transform=ccrs.PlateCarree(), shading="auto")
    ax.coastlines(linewidth=0.5, color="0.35")
    ax.set_extent([*DOMAIN["lon"], *DOMAIN["lat"]], crs=ccrs.PlateCarree())
    ax.set_title(f"{name}, {str(t0.values)[:16]}", fontsize=10)
    fig.colorbar(mesh, ax=ax, shrink=0.8, label=units)

## 5. The budget, in groups

`budget.groups` sums the terms as the reference's website file does:

tend = hflux + vflux + pe + cond + fric + ana + spte + **resid**

with tend = IKEVTEND (the change of the instantaneous column over the window), hflux = IHAKE + IHCV,
vflux = IVAKE + IVCV, pe = IPEKE, cond = all vapor-side physics, fric = all momentum-side physics,
ana = IAOKEV and spte = SPTE.

In [ ]:
g = budget.groups(case.sel(time=FRAME), "KEV").load()
fig = map_panels({name: g[name] for name in g.data_vars}, f"KEV budget, window centred {FRAME}")

How big is each group? RMS over the domain and the three days, and inside the AR mask if
`VKE_AR_MASKS` is set.

In [ ]:
g_all = budget.groups(case, "KEV")
mask = window_mask(case["time"])
where = "inside the IKEV AR mask" if mask is not None else "over the domain"
selection = region(g_all) if mask is None else region(g_all.where(mask))
rms = {name: float(np.sqrt(np.nanmean(selection[name].values ** 2))) for name in g_all.data_vars}

fig, ax = plt.subplots(figsize=(6, 3.2))
names = list(rms)
ax.barh(names, [rms[n] for n in names], color="#2a78d6", height=0.6)
ax.invert_yaxis()
ax.set_xlabel("RMS (kg s$^{-3}$)")
ax.set_title(f"KEV budget groups, RMS {where}", fontsize=10)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
print(f"RMS(resid) / RMS(tend) = {rms['resid'] / rms['tend']:.3f}")

## 6. Splitting the residual

The residual splits *exactly* into three parts, each a difference of quantities already computed:

| part | definition | what it measures |
|---|---|---|
| tendency mismatch | IKEVTEND − I_KEVTEND − SPTE | the change of the instantaneous column vs the integral of the local tendency, plus the boundary term |
| MERRA-2 closure | I_KEVTEND − (IDOKE + IDOV + cond + fric + ana) | how well MERRA-2's own tendencies close the budget |
| offline dynamics | (IDOKE + IDOV) − (hflux + vflux + pe) | our dynamics terms vs MERRA-2's own dynamics tendency |

The three add up to the residual (checked in `residual_parts`, to float32 rounding).

In [ ]:
def residual_parts(ds, g):
    physics = g["cond"] + g["fric"] + g["ana"]
    parts = xr.Dataset({
        "tendency mismatch": ds["IKEVTEND"] - ds["I_KEVTEND"] - ds["SPTE"],
        "MERRA-2 closure": ds["I_KEVTEND"] - (ds["IDOKE"] + ds["IDOV"] + physics),
        "offline dynamics": (ds["IDOKE"] + ds["IDOV"]) - (g["hflux"] + g["vflux"] + g["pe"]),
    })
    # The parts add up to the residual, to float32 rounding of the terms.
    gap = float(np.abs(sum(parts[p] for p in parts.data_vars) - g["resid"]).max())
    assert gap <= 1e-5 * float(np.abs(ds["IKEVTEND"]).max()), gap
    return parts


parts = residual_parts(case.sel(time=FRAME), g).load()
fig = map_panels({**{p: parts[p] for p in parts.data_vars}, "resid": g["resid"]},
                 f"KEV residual in three parts, {FRAME}", ncols=2)

parts_all = residual_parts(case, g_all)
selection = region(parts_all) if mask is None else region(parts_all.where(mask))
for p in parts_all.data_vars:
    print(f"{p:18s} RMS / RMS(tend) = {np.sqrt(np.nanmean(selection[p].values ** 2)) / rms['tend']:.3f}")

## 7. In the vertical

From the live window (section 2), area means over the domain (weighted by cos latitude) of the
offline dynamics against MERRA-2's own dynamics tendency, on each side of the budget. Same colors in
both panels: ours is blue and solid, MERRA-2's orange and dashed.

In [ ]:
def area_mean(da):
    da = region(da)
    weights = np.cos(np.deg2rad(da["lat"]))
    return da.weighted(weights).mean(("lat", "lon")).squeeze()


w = window.isel(time=0)
sides = {
    "momentum side": (w["HAKE"] + w["VAKE"] + w["PEKE"], w["DOKE"], "HAKE + VAKE + PEKE", "DOKE"),
    "vapor side": (w["HCV"] + w["VCV"], w["DOV"], "HCV + VCV", "DOV"),
}
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for ax, (side, (ours, merra, ours_label, merra_label)) in zip(axes, sides.items()):
    p_hpa = w["lev"] / 100
    # Ours solid, MERRA-2's dashed, so the two are told apart without color too.
    for profile, color, style, label in ((area_mean(ours), "#2a78d6", "-", ours_label),
                                         (area_mean(merra), "#eb6834", "--", merra_label)):
        ax.plot(profile, p_hpa, color=color, linestyle=style, linewidth=2, label=label)
    ax.axvline(0, color="0.8", linewidth=0.8)
    ax.set_title(side, fontsize=10)
    ax.set_xlabel("area mean (m$^2$ s$^{-3}$)")
    ax.legend(frameon=False, fontsize=8)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
axes[0].set_ylabel("pressure (hPa)")
axes[0].invert_yaxis()
fig.suptitle(f"Dynamics, ours vs MERRA-2's, window centred {str(w['time'].values)[:16]}")

## Your investigation starts here

Some threads to pull, each a change of one thing:

- **Where is the offline-dynamics error?** Maps and profiles of each term against DOKE / DOV; level by level
  with `keep_3d=True`.
- **Time averaging.** The terms are products of 3-hour *mean* fields; MERRA-2's tendencies are means of
  products. Compare with terms computed from the instantaneous fields at the two ends of a window.
- **Terrain.** Horizontal derivatives on pressure levels are missing next to below-ground points, so those
  integrals start higher in the column. Compare ocean and land points.
- **SPTE over the ocean.** Where p_s > 1000 hPa the column bottom is fixed, so the boundary term arguably
  should be zero there.
- **Continuity.** With `flux_form=True`, `IDIVV` and `IQKDIVV` measure how far the discrete winds are from
  continuity, which is what separates the advective and flux forms.